# Final 2024 Evaluation 01: Frozen Core Models

This notebook performs the one-time final test of one frozen model at each distinct prediction time. Model and feature selection used 2019 chronological folds, and 2023 was the outside development check. The 2024 outcomes were not used to choose a model, feature, parameter, or threshold.

Each model is refit on all 2019 rows with its documented configuration. The resulting probabilities are evaluated on 2024 once. Results are shown at the standard 0.50 threshold and at the operating threshold selected before 2024 was opened.

In [1]:
from pathlib import Path
from time import perf_counter

import catboost
import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler

AIRPORT = "JFK"
TRAIN_YEAR = 2019
TEST_YEAR = 2024
RANDOM_STATE = 42
N_JOBS = 4

FROZEN_THRESHOLDS = {
    "1A CatBoost 04": 0.31,
    "2A Logistic Regression 01": 0.22,
    "2B Logistic Regression 02": 0.39,
    "2C Logistic Regression 02": 0.45,
}

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
print(f"pandas {pd.__version__}; scikit-learn {sklearn.__version__}; CatBoost {catboost.__version__}")
pd.DataFrame([
    {"model": "1A", "notebook": "catboost_1a_04.ipynb", "prediction time": "before pushback", "threshold": 0.31},
    {"model": "2A", "notebook": "logistic_regression_2a_01.ipynb", "prediction time": "before pushback", "threshold": 0.22},
    {"model": "2B", "notebook": "logistic_regression_2b_02.ipynb", "prediction time": "at pushback", "threshold": 0.39},
    {"model": "2C", "notebook": "logistic_regression_2c_02.ipynb", "prediction time": "after takeoff", "threshold": 0.45},
]).set_index("model")

pandas 3.0.5; scikit-learn 1.9.0; CatBoost 1.2.10


,notebook,prediction time,threshold
model,,,
1A,catboost_1a_04.ipynb,before pushback,0.3100
2A,logistic_regression_2a_01.ipynb,before pushback,0.2200
2B,logistic_regression_2b_02.ipynb,at pushback,0.3900
2C,logistic_regression_2c_02.ipynb,after takeoff,0.4500


## Load and validate the final-test datasets

Model 1A uses the established full-history rotation, airport-wide 60-minute backlog, and same-airline 60-minute backlog datasets. They are paired in memory after row-identity checks. Models 2A, 2B, and 2C use the same arrival rows from the baseline arrival dataset.

In [2]:
def find_project_root(start: Path) -> Path:
    required = [
        Path("data/features/JFK_2019_arrivals.csv"),
        Path("data/features/JFK_2024_arrivals.csv"),
        Path("data/features/JFK_2019_departures_rotation_full_history.csv"),
        Path("data/features/JFK_2024_departures_rotation_full_history.csv"),
        Path("data/features/JFK_2019_departures_backlog_w60.csv"),
        Path("data/features/JFK_2024_departures_backlog_w60.csv"),
        Path("data/features/JFK_2019_departures_backlog_airline_w60.csv"),
        Path("data/features/JFK_2024_departures_backlog_airline_w60.csv"),
    ]
    for candidate in (start, *start.parents):
        if all((candidate / path).is_file() for path in required):
            return candidate
    raise FileNotFoundError(f"Could not locate the required final-evaluation datasets: {required}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
FEATURE_DIR = PROJECT_ROOT / "data/features"

arrival_frames = {
    year: pd.read_csv(FEATURE_DIR / f"{AIRPORT}_{year}_arrivals.csv", low_memory=False)
    for year in (TRAIN_YEAR, TEST_YEAR)
}
rotation_frames = {
    year: pd.read_csv(
        FEATURE_DIR / f"{AIRPORT}_{year}_departures_rotation_full_history.csv",
        low_memory=False,
    )
    for year in (TRAIN_YEAR, TEST_YEAR)
}
backlog_frames = {
    year: pd.read_csv(
        FEATURE_DIR / f"{AIRPORT}_{year}_departures_backlog_w60.csv",
        low_memory=False,
    )
    for year in (TRAIN_YEAR, TEST_YEAR)
}
airline_backlog_frames = {
    year: pd.read_csv(
        FEATURE_DIR / f"{AIRPORT}_{year}_departures_backlog_airline_w60.csv",
        low_memory=False,
    )
    for year in (TRAIN_YEAR, TEST_YEAR)
}

pd.DataFrame([
    {"dataset": "arrivals", "year": year, "rows": len(arrival_frames[year]), "columns": arrival_frames[year].shape[1]}
    for year in (TRAIN_YEAR, TEST_YEAR)
] + [
    {"dataset": name, "year": year, "rows": len(frames[year]), "columns": frames[year].shape[1]}
    for name, frames in [
        ("rotation full history", rotation_frames),
        ("airport backlog W60", backlog_frames),
        ("same-airline backlog W60", airline_backlog_frames),
    ]
    for year in (TRAIN_YEAR, TEST_YEAR)
]).set_index(["dataset", "year"])

rows  columns
dataset                  year                 
arrivals                 2019  107354      116
                         2024  104555      116
rotation full history    2019  107430      133
                         2024  104715      133
airport backlog W60      2019  107430      120
                         2024  104715      120
same-airline backlog W60 2019  107430      121
                         2024  104715      121

In [3]:
ROTATION_FEATURES = [
    "ROTATION_STATUS", "ROTATION_MATCH_FOUND", "ROTATION_INBOUND_ORIGIN",
    "ROTATION_SCHEDULED_TURN_MINUTES", "ROTATION_INBOUND_ARRIVED_BY_CUTOFF",
    "ROTATION_INBOUND_NOT_ARRIVED_BY_CUTOFF", "ROTATION_INBOUND_OVERDUE_MINUTES",
    "ROTATION_LOG_INBOUND_OVERDUE_MINUTES", "ROTATION_ACTUAL_TURN_MINUTES",
    "ROTATION_LOG_ACTUAL_TURN_MINUTES", "ROTATION_INBOUND_ARR_DELAY",
    "ROTATION_INBOUND_DELAYED_15", "ROTATION_LOG_SCHEDULED_TURN_MINUTES",
]
BACKLOG_FEATURES = [
    "BACKLOG_W60_PENDING_COUNT", "BACKLOG_W60_COMPLETED_COUNT",
    "BACKLOG_W60_MEAN_DEP_DELAY",
]
AIRLINE_BACKLOG_FEATURES = [
    "AIRLINE_BACKLOG_W60_PENDING_COUNT", "AIRLINE_BACKLOG_W60_COMPLETED_COUNT",
    "AIRLINE_BACKLOG_W60_MEAN_DEP_DELAY", "AIRLINE_BACKLOG_W60_DELAY_RATE",
    "AIRLINE_BACKLOG_W60_PENDING_SHARE",
]
FULL_BACKLOG_FEATURES = [
    "BACKLOG_W60_SCHEDULED_COUNT", "BACKLOG_W60_COMPLETED_COUNT",
    "BACKLOG_W60_PENDING_COUNT", "BACKLOG_W60_DELAYED_DEPARTURE_COUNT",
]
FULL_AIRLINE_BACKLOG_FEATURES = [
    "AIRLINE_BACKLOG_W60_SCHEDULED_COUNT", "AIRLINE_BACKLOG_W60_COMPLETED_COUNT",
    "AIRLINE_BACKLOG_W60_PENDING_COUNT", "AIRLINE_BACKLOG_W60_DELAYED_DEPARTURE_COUNT",
]
DEPARTURE_IDENTITY_COLUMNS = [
    "FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline",
    "Origin", "Dest", "CRSDepTime", "Tail_Number", "DepDel15",
]


def combine_departure_sources(year: int) -> pd.DataFrame:
    rotation = rotation_frames[year].copy()
    backlog = backlog_frames[year].copy()
    airline = airline_backlog_frames[year].copy()
    assert len(rotation) == len(backlog) == len(airline)
    pd.testing.assert_frame_equal(
        rotation[DEPARTURE_IDENTITY_COLUMNS], backlog[DEPARTURE_IDENTITY_COLUMNS], check_dtype=False
    )
    pd.testing.assert_frame_equal(
        rotation[DEPARTURE_IDENTITY_COLUMNS], airline[DEPARTURE_IDENTITY_COLUMNS], check_dtype=False
    )
    for frame in (rotation, backlog, airline):
        frame["FlightDate"] = pd.to_datetime(frame["FlightDate"], errors="raise")
        assert frame["FlightDate"].dt.year.eq(year).all()
        assert frame["Origin"].eq(AIRPORT).all()

    match = pd.to_numeric(rotation["ROTATION_MATCH_FOUND"], errors="raise")
    arrived = pd.to_numeric(rotation["ROTATION_INBOUND_ARRIVED_BY_CUTOFF"], errors="raise")
    not_arrived = pd.to_numeric(rotation["ROTATION_INBOUND_NOT_ARRIVED_BY_CUTOFF"], errors="raise")
    assert match.eq(arrived + not_arrived).all()
    actual_only = [
        "ROTATION_ACTUAL_TURN_MINUTES", "ROTATION_INBOUND_ARR_DELAY",
        "ROTATION_INBOUND_DELAYED_15",
    ]
    assert not rotation.loc[arrived.eq(0), actual_only].notna().any().any()

    scheduled, completed, pending, delayed = (
        pd.to_numeric(backlog[column], errors="raise") for column in FULL_BACKLOG_FEATURES
    )
    assert scheduled.eq(completed + pending).all() and delayed.le(completed).all()
    airline_scheduled, airline_completed, airline_pending, airline_delayed = (
        pd.to_numeric(airline[column], errors="raise") for column in FULL_AIRLINE_BACKLOG_FEATURES
    )
    assert airline_scheduled.eq(airline_completed + airline_pending).all()
    assert airline_delayed.le(airline_completed).all()

    combined = rotation.copy()
    for column in BACKLOG_FEATURES:
        combined[column] = backlog[column].to_numpy()
    for column in AIRLINE_BACKLOG_FEATURES:
        combined[column] = airline[column].to_numpy()
    return combined


departure_frames = {year: combine_departure_sources(year) for year in (TRAIN_YEAR, TEST_YEAR)}
print("Departure row identity and causal-state checks passed for 2019 and 2024.")

Departure row identity and causal-state checks passed for 2019 and 2024.


## Model 1A: CatBoost 04 before pushback

The frozen 41-field allowlist contains 20 baseline fields, 13 full-history rotation fields, three airport-wide backlog fields, and five same-airline backlog fields. Scheduled turns longer than 24 hours are masked exactly as in the selected experiment.

In [4]:
DEP_TARGET = "DepDel15"
DEP_CATEGORICAL = [
    "Month", "DayOfWeek", "Reporting_Airline", "Dest",
    "ROTATION_STATUS", "ROTATION_INBOUND_ORIGIN",
]
DEP_BASE_NUMERIC = [
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime", "Distance",
    "ASPM_PREVIOUS_SCHEDULED_DEPARTURES", "ASPM_PREVIOUS_SCHEDULED_ARRIVALS",
    "ASPM_CURRENT_SCHEDULED_DEPARTURES", "ASPM_CURRENT_SCHEDULED_ARRIVALS",
    "ASPM_NEXT_SCHEDULED_DEPARTURES", "ASPM_NEXT_SCHEDULED_ARRIVALS",
    "HourlyDewPointTemperature", "HourlyDryBulbTemperature",
    "HourlyPrecipitation", "HourlyRelativeHumidity", "HourlyVisibility", "HourlyWindSpeed",
]
DEP_ROTATION_NUMERIC = [field for field in ROTATION_FEATURES if field not in DEP_CATEGORICAL]
DEP_NUMERIC = [
    *DEP_BASE_NUMERIC, *DEP_ROTATION_NUMERIC,
    *BACKLOG_FEATURES, *AIRLINE_BACKLOG_FEATURES,
]
DEP_FEATURES = [*DEP_CATEGORICAL, *DEP_NUMERIC]
assert len(DEP_FEATURES) == 41 and len(DEP_CATEGORICAL) == 6 and len(DEP_NUMERIC) == 35


def prepare_departures(source: pd.DataFrame) -> pd.DataFrame:
    frame = source.copy()
    long_turn = pd.to_numeric(frame["ROTATION_SCHEDULED_TURN_MINUTES"], errors="coerce").gt(24 * 60)
    frame.loc[long_turn, ROTATION_FEATURES] = np.nan
    frame.loc[long_turn, "ROTATION_STATUS"] = "LONG_TURN_EXCLUDED"
    required = list(dict.fromkeys(["FlightDate", "CRSDepTime", DEP_TARGET, *DEP_FEATURES]))
    assert not set(required).difference(frame.columns)
    frame = frame[required].sort_values(["FlightDate", "CRSDepTime"], kind="stable").reset_index(drop=True)
    frame[DEP_TARGET] = pd.to_numeric(frame[DEP_TARGET], errors="raise").astype(int)
    for column in DEP_CATEGORICAL:
        frame[column] = frame[column].astype("string").fillna("MISSING").astype(str)
    for column in DEP_NUMERIC:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame


dep_train = prepare_departures(departure_frames[TRAIN_YEAR])
dep_test = prepare_departures(departure_frames[TEST_YEAR])
assert dep_train["FlightDate"].max() < dep_test["FlightDate"].min()

fit_start = perf_counter()
model_1a = CatBoostClassifier(
    iterations=683, depth=6, learning_rate=0.03, l2_leaf_reg=3,
    random_strength=1, loss_function="Logloss", eval_metric="PRAUC:type=Classic",
    random_seed=RANDOM_STATE, thread_count=N_JOBS,
    allow_writing_files=False, verbose=False,
)
model_1a.fit(dep_train[DEP_FEATURES], dep_train[DEP_TARGET], cat_features=DEP_CATEGORICAL, verbose=False)
fit_1a_seconds = perf_counter() - fit_start
predict_start = perf_counter()
prob_1a = model_1a.predict_proba(dep_test[DEP_FEATURES])[:, 1]
predict_1a_seconds = perf_counter() - predict_start
print(f"1A fit: {fit_1a_seconds:,.1f}s; 2024 prediction: {predict_1a_seconds:,.1f}s")

1A fit: 27.0s; 2024 prediction: 0.1s


## Models 2A, 2B, and 2C: frozen arrival ladder

All three models use the same arrival rows. Model 2A uses the 27-field baseline before pushback. Model 2B adds signed `DepDelay` and a degree-3 spline of the minutes remaining until scheduled arrival at pushback. Model 2C adds raw `TaxiOut`, actual takeoff clock cycles, and a degree-3 spline of the minutes remaining until scheduled arrival after takeoff. Each spline uses five quantile knots fitted only on 2019.

In [5]:
ARR_TARGET = "ArrDel15"
ARR_CATEGORICAL = ["Reporting_Airline", "Origin"]
ARR_BASE_NUMERIC = [
    "SCHED_DEP_TIME_SIN", "SCHED_DEP_TIME_COS", "SCHED_ARR_TIME_SIN", "SCHED_ARR_TIME_COS",
    "DAY_OF_WEEK_SIN", "DAY_OF_WEEK_COS", "DAY_OF_YEAR_SIN", "DAY_OF_YEAR_COS",
    "IS_WEEKEND", "CRSElapsedTime", "LOG_DISTANCE", "SCHEDULED_SPEED_PROXY",
    "ASPM_THREE_HOUR_SCHEDULED_DEPARTURES", "ASPM_THREE_HOUR_SCHEDULED_ARRIVALS",
    "ASPM_CURRENT_MINUS_PREVIOUS_TRAFFIC", "ASPM_NEXT_MINUS_CURRENT_TRAFFIC",
    "ASPM_MAX_HOURLY_TRAFFIC", "HourlyDryBulbTemperature", "TEMP_DEWPOINT_SPREAD",
    "LOG_PRECIPITATION", "HourlyVisibility", "WindX", "WindY", "ADVERSE_WEATHER",
    "NOAA_AGE_MINUTES",
]
ARR_2B_NUMERIC = [*ARR_BASE_NUMERIC, "DepDelay"]
ARR_2C_NUMERIC = [
    *ARR_BASE_NUMERIC, "DepDelay", "TaxiOut",
    "ACTUAL_TAKEOFF_TIME_SIN", "ACTUAL_TAKEOFF_TIME_COS",
]
MARGIN_2B = "MINUTES_TO_SCHEDULED_ARRIVAL_AT_PUSHBACK"
MARGIN_2C = "MINUTES_TO_SCHEDULED_ARRIVAL"
ARR_REQUIRED_SOURCE = [
    "FlightDate", "Dest", "CRSDepTime", ARR_TARGET,
    *ARR_CATEGORICAL, *ARR_2C_NUMERIC,
]


def prepare_arrivals(source: pd.DataFrame, year: int) -> pd.DataFrame:
    assert not set(ARR_REQUIRED_SOURCE).difference(source.columns)
    frame = source[ARR_REQUIRED_SOURCE].copy()
    frame["FlightDate"] = pd.to_datetime(frame["FlightDate"], errors="raise")
    assert frame["FlightDate"].dt.year.eq(year).all() and frame["Dest"].eq(AIRPORT).all()
    assert frame[ARR_TARGET].notna().all()
    frame = frame.sort_values(["FlightDate", "CRSDepTime"], kind="stable").reset_index(drop=True)
    frame[ARR_TARGET] = pd.to_numeric(frame[ARR_TARGET], errors="raise").astype(int)
    for column in ARR_CATEGORICAL:
        frame[column] = frame[column].astype(object)
    for column in set(ARR_2C_NUMERIC):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame[MARGIN_2B] = frame["CRSElapsedTime"] - frame["DepDelay"]
    frame[MARGIN_2C] = frame["CRSElapsedTime"] - frame["DepDelay"] - frame["TaxiOut"]
    assert frame[[MARGIN_2B, MARGIN_2C]].notna().all().all()
    return frame


arr_train = prepare_arrivals(arrival_frames[TRAIN_YEAR], TRAIN_YEAR)
arr_test = prepare_arrivals(arrival_frames[TEST_YEAR], TEST_YEAR)
assert arr_train["FlightDate"].max() < arr_test["FlightDate"].min()
assert len(ARR_CATEGORICAL) + len(ARR_BASE_NUMERIC) == 27
assert len(ARR_CATEGORICAL) + len(ARR_2B_NUMERIC) == 28
assert len(ARR_CATEGORICAL) + len(ARR_2C_NUMERIC) == 31


def make_arrival_pipeline(numeric_features, c_value, spline_feature=None):
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])
    transformers = [
        ("categorical", categorical_pipeline, ARR_CATEGORICAL),
        ("numeric", numeric_pipeline, numeric_features),
    ]
    if spline_feature is not None:
        spline_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("spline", SplineTransformer(
                n_knots=5, degree=3, knots="quantile",
                extrapolation="linear", include_bias=False,
            )),
            ("scaler", StandardScaler()),
        ])
        transformers.append(("schedule_margin", spline_pipeline, [spline_feature]))
    return Pipeline([
        ("preprocessor", ColumnTransformer(transformers=transformers, remainder="drop")),
        ("classifier", LogisticRegression(
            solver="liblinear", max_iter=5_000, random_state=RANDOM_STATE,
            C=c_value, l1_ratio=1.0, class_weight=None,
        )),
    ])


arrival_designs = {
    "2A Logistic Regression 01": {
        "numeric": ARR_BASE_NUMERIC, "spline": None, "C": 0.1,
        "source_fields": 27, "prepared_predictors": 98,
    },
    "2B Logistic Regression 02": {
        "numeric": ARR_2B_NUMERIC, "spline": MARGIN_2B, "C": 0.01,
        "source_fields": 28, "prepared_predictors": 105,
    },
    "2C Logistic Regression 02": {
        "numeric": ARR_2C_NUMERIC, "spline": MARGIN_2C, "C": 0.1,
        "source_fields": 31, "prepared_predictors": 108,
    },
}

In [6]:
arrival_models = {}
arrival_probabilities = {}
timing_rows = [{
    "model": "1A CatBoost 04", "fit_seconds": fit_1a_seconds,
    "test_predict_seconds": predict_1a_seconds, "source_fields": 41,
    "prepared_predictors": 41,
}]

for model_name, design in arrival_designs.items():
    input_columns = [*ARR_CATEGORICAL, *design["numeric"]]
    if design["spline"] is not None:
        input_columns.append(design["spline"])
    model = make_arrival_pipeline(design["numeric"], design["C"], design["spline"])
    fit_start = perf_counter()
    model.fit(arr_train[input_columns], arr_train[ARR_TARGET])
    fit_seconds = perf_counter() - fit_start
    prepared_count = len(model.named_steps["preprocessor"].get_feature_names_out())
    assert prepared_count == design["prepared_predictors"], (model_name, prepared_count)
    predict_start = perf_counter()
    probabilities = model.predict_proba(arr_test[input_columns])[:, 1]
    predict_seconds = perf_counter() - predict_start
    arrival_models[model_name] = model
    arrival_probabilities[model_name] = probabilities
    timing_rows.append({
        "model": model_name, "fit_seconds": fit_seconds,
        "test_predict_seconds": predict_seconds,
        "source_fields": design["source_fields"],
        "prepared_predictors": prepared_count,
    })
    print(f"{model_name}: fit {fit_seconds:,.1f}s; 2024 prediction {predict_seconds:,.1f}s")

timing_results = pd.DataFrame(timing_rows).set_index("model")
timing_results

2A Logistic Regression 01: fit 7.0s; 2024 prediction 0.1s


2B Logistic Regression 02: fit 0.8s; 2024 prediction 0.1s


2C Logistic Regression 02: fit 6.1s; 2024 prediction 0.1s


,fit_seconds,test_predict_seconds,source_fields,prepared_predictors
model,,,,
1A CatBoost 04,26.9566,0.0690,41,41
2A Logistic Regression 01,6.9949,0.0692,27,98
2B Logistic Regression 02,0.7949,0.0789,28,105
2C Logistic Regression 02,6.0545,0.0848,31,108


## Final test results

Average precision, ROC AUC, and Brier score use the probabilities directly. The classification measures are reported at both 0.50 and the frozen training-selected threshold. No result in this section changes the selected models or thresholds.

In [7]:
def ranking_metrics(model_name, y_true, probabilities):
    return {
        "model": model_name,
        "test_rows": len(y_true),
        "test_delay_rate": np.mean(y_true),
        "average_precision": average_precision_score(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "brier_score": brier_score_loss(y_true, probabilities),
    }


def operating_metrics(model_name, y_true, probabilities, threshold_policy, threshold):
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "threshold_policy": threshold_policy,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "mcc": matthews_corrcoef(y_true, predictions),
        "predicted_positive_rate": predictions.mean(),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


probability_sets = {
    "1A CatBoost 04": (dep_test[DEP_TARGET], prob_1a),
    **{
        name: (arr_test[ARR_TARGET], probabilities)
        for name, probabilities in arrival_probabilities.items()
    },
}
ranking_results = pd.DataFrame([
    ranking_metrics(name, y_true, probabilities)
    for name, (y_true, probabilities) in probability_sets.items()
]).set_index("model")

operating_results = pd.DataFrame([
    operating_metrics(name, y_true, probabilities, policy, threshold)
    for name, (y_true, probabilities) in probability_sets.items()
    for policy, threshold in [
        ("Default", 0.50),
        ("Frozen training-selected F1", FROZEN_THRESHOLDS[name]),
    ]
]).set_index(["model", "threshold_policy"])

display(ranking_results)
display(operating_results)

,test_rows,test_delay_rate,average_precision,roc_auc,brier_score
model,,,,,
1A CatBoost 04,104715,0.2038,0.7250,0.8517,0.0983
2A Logistic Regression 01,104555,0.2161,0.3482,0.6569,0.1603
2B Logistic Regression 02,104555,0.2161,0.8757,0.9254,0.0626
2C Logistic Regression 02,104555,0.2161,0.9268,0.9611,0.0465


threshold  accuracy  \
model                     threshold_policy                                   
1A CatBoost 04            Default                         0.5000    0.8730   
                          Frozen training-selected F1     0.3100    0.8670   
2A Logistic Regression 01 Default                         0.5000    0.7854   
                          Frozen training-selected F1     0.2200    0.6520   
2B Logistic Regression 02 Default                         0.5000    0.9223   
                          Frozen training-selected F1     0.3900    0.9201   
2C Logistic Regression 02 Default                         0.5000    0.9402   
                          Frozen training-selected F1     0.4500    0.9401   

                                                       balanced_accuracy  \
model                     threshold_policy                                 
1A CatBoost 04            Default                                 0.7133   
                          Frozen training-selected F1             0.7530   
2A Logistic Regression 01 Default                                 0.5105   
                          Frozen training-selected F1             0.6097   
2B Logistic Regression 02 Default                                 0.8428   
                          Frozen training-selected F1             0.8535   
2C Logistic Regression 02 Default                                 0.8853   
                          Frozen training-selected F1             0.8905   

                                                       precision  recall  \
model                     threshold_policy                                 
1A CatBoost 04            Default                         0.8694  0.4437   
                          Frozen training-selected F1     0.7246  0.5605   
2A Logistic Regression 01 Default                         0.5768  0.0262   
                          Frozen training-selected F1     0.3184  0.5351   
2B Logistic Regression 02 Default                         0.9187  0.7029   
                          Frozen training-selected F1     0.8740  0.7362   
2C Logistic Regression 02 Default                         0.9238  0.7885   
                          Frozen training-selected F1     0.9091  0.8031   

                                                          f1    mcc  \
model                     threshold_policy                            
1A CatBoost 04            Default                     0.5875 0.5630   
                          Frozen training-selected F1 0.6321 0.5593   
2A Logistic Regression 01 Default                     0.0502 0.0873   
                          Frozen training-selected F1 0.3993 0.1877   
2B Logistic Regression 02 Default                     0.7964 0.7597   
                          Frozen training-selected F1 0.7992 0.7540   
2C Logistic Regression 02 Default                     0.8508 0.8177   
                          Frozen training-selected F1 0.8528 0.8179   

                                                       predicted_positive_rate  \
model                     threshold_policy                                       
1A CatBoost 04            Default                                       0.1040   
                          Frozen training-selected F1                   0.1577   
2A Logistic Regression 01 Default                                       0.0098   
                          Frozen training-selected F1                   0.3632   
2B Logistic Regression 02 Default                                       0.1653   
                          Frozen training-selected F1                   0.1820   
2C Logistic Regression 02 Default                                       0.1844   
                          Frozen training-selected F1                   0.1909   

                                                          tn     fp     fn  \
model                     threshold_policy                                   
1A CatBoost 04            Default                      81949   1

In [8]:
documented_2023 = pd.DataFrame([
    {"model": "1A CatBoost 04", "average_precision_2023": 0.7526, "roc_auc_2023": 0.8546, "brier_score_2023": 0.1086, "f1_2023": 0.6511, "mcc_2023": 0.5640},
    {"model": "2A Logistic Regression 01", "average_precision_2023": 0.4019, "roc_auc_2023": 0.6744, "brier_score_2023": 0.1741, "f1_2023": 0.4425, "mcc_2023": 0.2153},
    {"model": "2B Logistic Regression 02", "average_precision_2023": 0.8704, "roc_auc_2023": 0.9150, "brier_score_2023": 0.0748, "f1_2023": 0.7887, "mcc_2023": 0.7357},
    {"model": "2C Logistic Regression 02", "average_precision_2023": 0.9145, "roc_auc_2023": 0.9491, "brier_score_2023": 0.0594, "f1_2023": 0.8324, "mcc_2023": 0.7885},
]).set_index("model")

frozen_2024 = operating_results.xs("Frozen training-selected F1", level="threshold_policy")
comparison_2023_2024 = documented_2023.join(
    ranking_results.rename(columns={
        "average_precision": "average_precision_2024",
        "roc_auc": "roc_auc_2024",
        "brier_score": "brier_score_2024",
    })[["average_precision_2024", "roc_auc_2024", "brier_score_2024"]]
).join(
    frozen_2024.rename(columns={"f1": "f1_2024", "mcc": "mcc_2024"})[["f1_2024", "mcc_2024"]]
)
for metric in ["average_precision", "roc_auc", "brier_score", "f1", "mcc"]:
    comparison_2023_2024[f"{metric}_change"] = (
        comparison_2023_2024[f"{metric}_2024"] - comparison_2023_2024[f"{metric}_2023"]
    )
comparison_2023_2024

,average_precision_2023,roc_auc_2023,brier_score_2023,f1_2023,mcc_2023,average_precision_2024,roc_auc_2024,brier_score_2024,f1_2024,mcc_2024,average_precision_change,roc_auc_change,brier_score_change,f1_change,mcc_change
model,,,,,,,,,,,,,,,
1A CatBoost 04,0.7526,0.8546,0.1086,0.6511,0.5640,0.7250,0.8517,0.0983,0.6321,0.5593,-0.0276,-0.0029,-0.0103,-0.0190,-0.0047
2A Logistic Regression 01,0.4019,0.6744,0.1741,0.4425,0.2153,0.3482,0.6569,0.1603,0.3993,0.1877,-0.0537,-0.0175,-0.0138,-0.0432,-0.0276
2B Logistic Regression 02,0.8704,0.9150,0.0748,0.7887,0.7357,0.8757,0.9254,0.0626,0.7992,0.7540,0.0053,0.0104,-0.0122,0.0105,0.0183
2C Logistic Regression 02,0.9145,0.9491,0.0594,0.8324,0.7885,0.9268,0.9611,0.0465,0.8528,0.8179,0.0123,0.0120,-0.0129,0.0204,0.0294


## Interpretation rule

These values are final generalization estimates for the four previously selected designs. They may reveal performance drift between 2023 and 2024, but they are not a new validation set. No model, allowlist, transformation, parameter, or operating threshold will be changed in response to this table.